In [ ]:


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import numpy as np
#from sklearn.decomposition import PCA
#from sklearn.neighbors import NearestNeighbors
#import igraph as ig
#import leidenalg as la
#from umap.umap_ import fuzzy_simplicial_set
#import umap
import os
import math
#import pickle
from natsort import natsorted
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')
#import tensorflow as tf


import sys
import pickle

#sys.path.append('/home/mystique/Banksy_py')
#sys.path.append('/home/mystique/Banksy_py/banksy')
#sys.path.append('/home/mystique/Banksy_py/banksy_utils')


sys.path.append('/home/shamini_pathomics_io/Banksy_py')
sys.path.append('/home/shamini_pathomics_io/Banksy_py/banksy')
sys.path.append('/home/shamini_pathomics_io/Banksy_py/banksy_utils')


In [ ]:
home_dir = Path.home()

### GCP
working_dir = home_dir / 'ext_hd_sammy' / 'projects' 
out_dir = working_dir / 'out' / 'out_stomics'

### HPC
#working_dir = home_dir / 'scratch' / 'projects' / 'sammy'
#out_dir = working_dir / 'out' / 'out_stomics'


src_dir = out_dir / 'script02_output/' / 'cellbin'

dst_dir = out_dir / 'script03_output/'
os.makedirs(dst_dir, exist_ok=True)

#working_dir = working_dir.resolve()
working_dir


In [ ]:
adata_filenames = [file for file in os.listdir(src_dir) if file.endswith('.h5ad')]
adata_filenames = natsorted(adata_filenames)
adata_filenames

In [ ]:
adata = sc.read_h5ad(os.path.join(str(src_dir), adata_filenames[0]))


In [ ]:
# Plot
sns.scatterplot(
    data=adata.obs,
    x='x',
    y='y',
    s=.15,  # marker size

)


In [ ]:

### subset 10000-15000 by coordinates
adata = adata[adata.obs['x'] > 5000]
adata = adata[adata.obs['x'] < 12500]
adata = adata[adata.obs['y'] > 2500]
adata = adata[adata.obs['y'] < 10000]



In [ ]:

### subset 10000-15000 by coordinates
adata = adata[adata.obs['x'] > 12500]
adata = adata[adata.obs['x'] < 20000]
adata = adata[adata.obs['y'] > 2500]
adata = adata[adata.obs['y'] < 10000]


In [ ]:
sns.scatterplot(data=adata.obs, x='x', y='y', s=.45, hue='leiden')

In [ ]:
'''
1. FIRST WE WILL PERFORM A NEAREST NEIGHBOR BASED DISTANCE CALCULATION TO COMPUTE THE NECESSARY DISTANCES BETWEEN THE CELLS
'''

from banksy.main import median_dist_to_nearest_neighbour
from banksy.initialize_banksy import initialize_banksy

adatas = []
banksy_list = []

# set params
# ==========
plot_graph_weights = True
k_geom = 30 # only for fixed type
max_m = 1 # azumithal transform up to kth order
nbr_weight_decay = "scaled_gaussian" # can also be "reciprocal", "uniform" or "ranked"

for filename in adata_filenames:

    adata = sc.read_h5ad(os.path.join(str(src_dir), filename))
    adatas.append(adata)
    adata.obsm['spatial'] = adata.obs[['x', 'y']].values
    print(adata.shape)
    adata = adata[:,adata.var['highly_variable']]
    adatas.append(adata)

    # Find median distance to closest neighbours, the median distance will be `sigma`
    nbrs = median_dist_to_nearest_neighbour(adata, key = 'spatial')


    banksy_dict = initialize_banksy(
        adata,
        ('x', 'y', 'spatial'),
        k_geom,
        nbr_weight_decay=nbr_weight_decay,
        max_m=max_m,
        plt_edge_hist=False,
        plt_nbr_weights=False,
        plt_agf_angles=False, # takes long time to plot
        plt_theta=False,
    )

    banksy_list.append(banksy_dict)

### remove all the warnings and messages from the output

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
import gc
gc.collect()

In [ ]:
'''
2. NEXT WE WILL CONSTRUCT A BANKSY MATRIX
'''

from banksy.embed_banksy import generate_banksy_matrix

### the following are the main hyperparamters for the banksy algorithm
### ------------------------------------------------------------------

pca_dims = [21] ### Dimensionality to which to reduce data to
#lamda_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0] ### list of lamda values, setting higher value will result in more domain specific clustering
lamda_list = [0.8]
### ------------------------------------------------------------------
### the following are the main hyperparamters for the banksy algorithm
### ------------------------------------------------------------------


banksy_matrix_list = []

for i, banksy_dict in enumerate(banksy_list):
    adata = adatas[i]    
    banksy_dict, banksy_matrix = generate_banksy_matrix(adata, banksy_dict, lamda_list, max_m, verbose=False)
    banksy_matrix_list.append(banksy_matrix)
    banksy_list[i] = banksy_dict

In [ ]:
### append non-spatial results to the banksy_dict for comparison

from banksy.main import concatenate_all

for i, banksy_dict in enumerate(banksy_list):
    adata = adatas[i]
    banksy_dict['nonspatial'] = {### here we append the non-spatial matrix (adata.X) to obtain the non-spatial clustering results
        0.0: {"adata": concatenate_all([adata.X], 0, adata=adata), }
        }
    banksy_list[i] = banksy_dict


In [ ]:
'''
3. BANKSY APPLIES PCA AND UMAP OVER THE SPATIAL DERIVED MATRIX, FOLLOWING BY LEIDEN CLUSTERING
'''

import gc
gc.collect()

from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition

results_df_list = []
max_num_labels_list = []

for i, banksy_dict in enumerate(banksy_list):
    adata = adatas[i]
    
  
    pca_umap(banksy_dict,
             pca_dims=pca_dims,
             add_umap=False,
             plt_remaining_var=False,
             verbose=False)
    

    seed=329
    
    
    #resolutions = [0.45, 0.9, 1.5] ### clustering resolution for umap
    resolutions = [#.1, 
                   .15, 
                   #.3, 
                   #.4, 
                   #.5, 
                   #.9
                   ]

    results_df, max_num_labels = run_Leiden_partition(
        banksy_dict,
        resolutions,
        num_nn = 25,
        num_iterations = -1,
        partition_seed = seed,
        match_labels = True,
        verbose = False
    )
        
    results_df_list.append(results_df)
    max_num_labels_list.append(max_num_labels)
    
    p_names = results_df.index
    
    
    for p_name in p_names:
        labels = results_df.loc[p_name, 'relabeled']
        adata_results = results_df.loc[p_name, "adata"]
        adata_results

        #pc_temp = adata_results.obsm(f"reduced_pc {pca_dims[0]}")
        #pca_umap = adata_results.obsm(f"umap {pca_dims[0]}")

        label_name = f"labels_{p_name}"
        label_name

        print(label_name)
        adata_results.obs[label_name] = np.char.mod('%d', labels.dense)
        adata_results.obs[label_name] = adata_results.obs[label_name].astype('category')
        adata.obs = adata.obs.reindex(adata_results.obs.index)
        adata.obs[label_name] = adata_results.obs[label_name]

    adata.obsm[f'pc{pca_dims[0]}_banksy'] = adata_results.obsm[f'reduced_pc_{pca_dims[0]}'].copy()
    #adata.obsm[f'umap{pca_dims[0]}_banksy'] = adata_results.obsm[f'reduced_pc_{pca_dims[0]}_umap'].copy()
    
    sample_name = banksy_dict['nonspatial'][0.0]['adata'].obs['sample_id'][0]

    with open(str(dst_dir) + f'banksy_results_{sample_name}.pkl', 'wb') as f:
        pickle.dump(banksy_dict, f)


    with open(str(dst_dir) + f'banksy_results_{sample_name}_results_df.pkl', 'wb') as f:
        pickle.dump(results_df, f)

    with open(str(dst_dir) + f'banksy_results_{sample_name}_max_num_labels.pkl', 'wb') as f:
        pickle.dump(max_num_labels, f)

    with open(str(dst_dir) + f'banksy_results_{sample_name}_p_names.pkl', 'wb') as f:
        pickle.dump(p_names, f)
    
    adata.write_h5ad(dst_dir / f'banksy_results_{sample_name}_adata.h5ad')


    #banksy_list[i] = banksy_dict

warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
adata

In [ ]:
sns.scatterplot(data=adata.obs, x='x', y='y', hue='labels_scaled_gaussian_pc21_nc0.80_r0.15', s=1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)